In [ ]:
pip install scikit-posthocs

Loading and inspecting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import scikit_posthocs as sp
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv("WineQT.csv")
df.shape
df.info()
df.head()
df.describe()


Checking data quality

1.1 missing values

In [ ]:
df.isna().sum()

1.2 Checking duplicates

In [ ]:
df.duplicated().sum()

1.3 Dropping ID col
```
 Col Id is an identifier
 ```

In [ ]:
df.drop(columns="Id", inplace=True)

In [ ]:
df.head()

Checking skewness

In [ ]:
df.skew()

In [ ]:
summary = df.describe().T
summary["median"] = df.median()
summary["mean_minus_median"] = summary["mean"] - summary["median"]
summary

In [ ]:
features = df.drop(columns=["quality"]).columns

fig, axes = plt.subplots(4, 3, figsize=(15, 14))
for ax, col in zip(axes.flatten(), features):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.axvline(df[col].mean(), color="red", linestyle="--", label="mean")
    ax.axvline(df[col].median(), color="green", linestyle="-", label="median")
    ax.set_title(f"{col} (skew={df[col].skew():.2f})")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 12))
for ax, col in zip(axes.flatten(), features):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
def count_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()

df[features].apply(count_outliers).sort_values(ascending=False)

Examining the target

In [ ]:
counts = df["quality"].value_counts().sort_index()
percent = (df["quality"].value_counts(normalize=True).sort_index() * 100).round(1)

pd.DataFrame({"count": counts, "percent": percent})

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="quality", data=df, order=sorted(df["quality"].unique()))
plt.title("Distribution of wine quality scores")
plt.show()

In [ ]:
df.corr()

In [ ]:
df.corr(method="spearman")

In [ ]:
low = df.loc[df["quality"] <= 4, "alcohol"]
high = df.loc[df["quality"] >= 7, "alcohol"]
print(len(low), len(high))
print(low.mean(), high.mean())
print(low.median(), high.median())

Normality check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
stats.probplot(low, dist="norm", plot=axes[0]);  axes[0].set_title("Low quality")
stats.probplot(high, dist="norm", plot=axes[1]); axes[1].set_title("High quality")

print(stats.shapiro(low))
print(stats.shapiro(high))

In [ ]:
print(stats.levene(low, high))

In [ ]:
# Welch's t-test
print(stats.ttest_ind(low, high, equal_var=False))

# Mann-Whitney U
print(stats.mannwhitneyu(low, high, alternative="two-sided"))

In [ ]:
groups = [g["alcohol"].values for _, g in df.groupby("quality")]

print(stats.f_oneway(*groups))   # ANOVA
print(stats.kruskal(*groups))    # Kruskal-Wallis

In [ ]:
sp.posthoc_dunn(df, val_col="alcohol", group_col="quality", p_adjust="bonferroni")

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_sd = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (b.mean() - a.mean()) / pooled_sd

d = cohens_d(low, high)
print(d)